In [ ]:
from one.api import ONE
from labdata import *
from labdata.schema import *
from labdata import chronic_paper as cp
from labdata.rules import process_upload_jobs
from pathlib import Path
import shutil

one = ONE()

In [ ]:
def aws_upload(key,assigned_files, rule = 'ephys'):
    print(f'key: {key}')
    assigned = copy_to_upload_server(assigned_files,
                                     local_path = prefs['local_paths'][0],
                                     server_path = prefs['local_paths'][0],
                                     overwrite = True,
                                     job_host = 'manual',
                                     job_rule = rule,
                                     parse_filename = False,
                                     **key)
    process_upload_jobs(dict(job_id = assigned[0]['job_id']),
                            job_host = 'manual',prefs = prefs)

In [ ]:
#cp.IBLMatchedInsertion().to_ephys_session(repopulate=True)
EphysRecording() & 'subject_name LIKE "_ZM%"'

In [ ]:
pids_to_add = (cp.IBLMatchedInsertion() - cp.IBLMatchedInsertion().EphysRecording().proj()).fetch('pid') # only get pids that haven't been added to EphysRecording yet
#pids_to_add = cp.IBLMatchedInsertion().fetch('pid')
pids_to_add

In [ ]:
def move_file(file, filename_only=False):
    newpath = list(file.parts)
    newpath[-6] = f'_{newpath[-6]}' # add underscore to subject name (ONLY FOR NON LAB MICE)
    newpath = Path(*newpath)
    newpath = Path(prefs['local_paths'][0]) / Path(*newpath.parts[-6:])
    if '.imec.' in str(newpath):
        newpath = newpath.with_name(newpath.name.replace('.imec.', '.imec0.'))
    print(f'old path: {file} \nnew path: {newpath}')
    if filename_only:
        return newpath
    # move the file
    newpath.parent.mkdir(parents=True, exist_ok=True)
    # check if files are the same
    if newpath.exists() and newpath.stat().st_size == file.stat().st_size:
        print(f'File already exists: {newpath}')
    else:
        shutil.copy(file, newpath)
    return newpath

In [ ]:
for pid in pids_to_add:
    eid, prb = one.pid2eid(pid)
    all_pids, all_prbs = one.eid2pid(eid)
    session_dict = one.eid2ref(eid)

    # 1. insert the subject if its not in database (add an underscore before)
    alyx_sub = one.alyx.rest('subjects','read',id=session_dict['subject'])
    subject_dict = dict(subject_name=f'_{session_dict["subject"]}',
                        subject_dob=alyx_sub['birth_date'],
                        subject_sex=alyx_sub['sex'],
                        user_name='mmelin',
                        strain_name='C57BL/6J',)
    Subject().insert1(subject_dict, skip_duplicates=True)

    # 2. check if the session exists in the database
    key = dict(subject_name = subject_dict['subject_name'],
    		   session_name = f'{session_dict["date"]}/00{session_dict["sequence"]}',
    		   dataset_name = 'ephys',
               session_datetime = datetime.combine(session_dict['date'], datetime.min.time()))
    print(key)
    #if len(Session() & key) >=1:
    #    print(f'Session {key["session_name"]} already exists in the database')
    #    continue

    # 3. Download session data
    filepaths = []
    for prb in all_prbs:
        metapath = one.load_dataset(eid, '*.ap.meta', collection=f'raw_ephys_data/{prb}', download_only=True, check_hash=False)
        chpath = one.load_dataset(eid, '*.ap.ch', collection=f'raw_ephys_data/{prb}', download_only=True, check_hash=False)
        binpath = one.load_dataset(eid, '*.ap.cbin', collection=f'raw_ephys_data/{prb}', download_only=True, check_hash=False)
        filepaths.extend([metapath, chpath, binpath])
        #filepaths.extend([metapath, chpath,])

    new_filepaths = [move_file(fp, filename_only=True) for fp in filepaths]

    ## 4. insert the ephys session
    aws_upload(key,new_filepaths, rule = 'ephys')
